# LegalEagle — Extractor Agent + Risk Scorer Agent

Notebook 4 — Two cooperative AI agents:

1. **Extractor Agent** — wraps the fine-tuned BERT NER model as a LangChain Tool and extracts all legal entities from any contract
2. **Risk Scorer Agent** — combines extracted entities + Qdrant RAG results and scores each clause category 1-10 with reasoning

**Flow:** Contract text → Extractor Agent → entities → Risk Scorer Agent → risk report

## 0 — Imports

In [1]:
import warnings, json, re
import numpy as np
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    pipeline as hf_pipeline
)
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

print('All imports OK!')

All imports OK!


## 1 — Load Fine-tuned BERT NER Model

We load the model we trained in Notebook 1 from `models/bert-ner-cuad-final/`.

The model predicts BIO tags for 8 legal clause types:
Parties, Agreement_Date, Governing_Law, Termination, Indemnification, Confidentiality, IP_Ownership, Non_Compete

In [2]:
MODEL_PATH = Path('../models/bert-ner-cuad-final')

# Label map — must match how notebook 1 trained the model
LABELS = [
    'O',
    'B-Parties',          'I-Parties',
    'B-Agreement_Date',   'I-Agreement_Date',
    'B-Governing_Law',    'I-Governing_Law',
    'B-Termination',      'I-Termination',
    'B-Indemnification',  'I-Indemnification',
    'B-Confidentiality',  'I-Confidentiality',
    'B-IP_Ownership',     'I-IP_Ownership',
    'B-Non_Compete',      'I-Non_Compete',
]
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

print('Loading tokenizer...')
ner_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))

print('Loading model...')
ner_model = AutoModelForTokenClassification.from_pretrained(
    str(MODEL_PATH),
    num_labels=len(LABELS),
    ignore_mismatched_sizes=True
)
ner_model.eval()

print(f'Model loaded! Parameters: {sum(p.numel() for p in ner_model.parameters()):,}')
print(f'Labels: {LABELS}')

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded! Parameters: 108,904,721
Labels: ['O', 'B-Parties', 'I-Parties', 'B-Agreement_Date', 'I-Agreement_Date', 'B-Governing_Law', 'I-Governing_Law', 'B-Termination', 'I-Termination', 'B-Indemnification', 'I-Indemnification', 'B-Confidentiality', 'I-Confidentiality', 'B-IP_Ownership', 'I-IP_Ownership', 'B-Non_Compete', 'I-Non_Compete']


## 2 — NER Inference Function

Converts raw text into structured entities by:
1. Tokenizing into 256-token windows (with overlap)
2. Running BERT forward pass → argmax → label IDs
3. Merging consecutive B-/I- tags back into full entity strings

In [3]:
MAX_LEN = 256
STRIDE  = 128

def run_ner(text: str) -> dict:
    """Run NER on text. Returns dict of entity_type -> [list of spans]."""
    encoding = ner_tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length'
    )
    offset_maps = encoding.pop('offset_mapping')
    encoding.pop('overflow_to_sample_mapping', None)

    all_preds = []
    all_offsets = []

    with torch.no_grad():
        for i in range(encoding['input_ids'].shape[0]):
            chunk = {k: v[i:i+1] for k, v in encoding.items()}
            out   = ner_model(**chunk)
            preds = torch.argmax(out.logits, dim=-1)[0].tolist()
            all_preds.append(preds)
            all_offsets.append(offset_maps[i].tolist())

    # Decode predictions -> spans
    entities = {}
    current_label = None
    current_start = None

    for preds, offsets in zip(all_preds, all_offsets):
        for pred_id, (start, end) in zip(preds, offsets):
            if start == 0 and end == 0:
                continue  # skip [CLS]/[SEP]/padding
            label = ID2LABEL.get(pred_id, 'O')
            if label.startswith('B-'):
                if current_label and current_start is not None:
                    span = text[current_start:end].strip()
                    if span:
                        entities.setdefault(current_label, []).append(span)
                current_label = label[2:]
                current_start = start
            elif label.startswith('I-') and current_label == label[2:]:
                pass  # continue the span
            else:
                if current_label and current_start is not None:
                    span = text[current_start:start].strip()
                    if span:
                        entities.setdefault(current_label, []).append(span)
                current_label = None
                current_start = None

    return entities

# Quick test
test_text = (
    'This Agreement is entered into as of January 1 2020 by and between '
    'Company A and Company B. Either party may terminate upon 90 days notice. '
    'This agreement shall be governed by the laws of New York.'
)
sample = run_ner(test_text)
print('NER test result:')
for k, v in sample.items():
    print(f'  {k}: {v}')

NER test result:


## 3 — Extractor Agent: Wrap NER as a LangChain Tool

A **LangChain Tool** is a callable with a name, description, and input schema.
The agent decides *when* to call this tool based on the task description.

```
Extractor Agent
     │
     ├─► [Tool: extract_legal_entities] ─► BERT NER ─► {entities}
     │
     └─► Returns structured extraction report
```

In [4]:
@tool
def extract_legal_entities(contract_text: str) -> str:
    """Extract legal entities (Parties, Dates, Governing Law, Termination,
    Indemnification, Confidentiality, IP Ownership, Non-Compete) from a
    contract. Input is raw contract text. Returns JSON string of entities."""
    entities = run_ner(contract_text)
    # Deduplicate each list
    clean = {k: list(dict.fromkeys(v)) for k, v in entities.items()}
    return json.dumps(clean, indent=2)


# ── The Extractor Agent ──────────────────────────────────────────────────
class ExtractorAgent:
    """Agent that calls the BERT NER tool to extract entities from a contract."""

    def __init__(self):
        self.tool = extract_legal_entities
        self.tool_name = extract_legal_entities.name

    def run(self, contract_text: str) -> dict:
        print(f'[ExtractorAgent] Calling tool: {self.tool_name}')
        raw_output = self.tool.invoke(contract_text)
        entities = json.loads(raw_output)
        print(f'[ExtractorAgent] Extracted {len(entities)} entity types')
        return entities


extractor = ExtractorAgent()
print(f'Tool name        : {extract_legal_entities.name}')
print(f'Tool description : {extract_legal_entities.description[:80]}...')

Tool name        : extract_legal_entities
Tool description : Extract legal entities (Parties, Dates, Governing Law, Termination,
    Indemnif...


## 4 — Connect to Qdrant (from Notebook 3)

The Risk Scorer Agent needs RAG context — it searches Qdrant for similar
clauses from other contracts to compare against.

In [5]:
import subprocess, time

QDRANT_PORT = 6333
COLLECTION  = 'contracts'

def connect_qdrant():
    # Try Docker first
    try:
        check = subprocess.run(
            ['docker', 'ps', '--filter', 'name=qdrant-legal', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if 'qdrant-legal' in check.stdout:
            c = QdrantClient(host='localhost', port=QDRANT_PORT)
            info = c.get_collection(COLLECTION)
            print(f'Connected to Qdrant (Docker). Vectors: {info.points_count}')
            return c
    except Exception as e:
        print(f'Docker Qdrant not available: {e}')

    # Fall back to in-memory (re-indexing required)
    print('Using in-memory Qdrant — run Notebook 3 first for persistence!')
    return QdrantClient(':memory:')

qdrant_client = connect_qdrant()

embedder = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True}
)
print('Embedder ready!')

Connected to Qdrant (Docker). Vectors: 1602


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder ready!


## 5 — Load LLM for Risk Scoring

`flan-t5-base` (already cached from Notebook 3) will be used by the Risk Scorer Agent.

In [6]:
print('Loading LLM...')
gen = hf_pipeline(
    'text-generation',
    model='google/flan-t5-base',
    max_new_tokens=300,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=gen)
print('LLM ready!')

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'Blender

LLM ready!


## 6 — Risk Scorer Agent

**Flow:**
```
entities (from Extractor Agent)
        │
        ├─► For each clause type:
        │       ├─► Qdrant RAG: find similar clauses from other contracts
        │       └─► LLM prompt: compare + assign risk score 1-10
        │
        └─► Returns full risk report {clause: {score, reasoning, evidence}}
```

**Risk Scale:**
| Score | Meaning |
|---|---|
| 1-3 | Low risk — standard/favourable clause |
| 4-6 | Medium risk — needs review |
| 7-10 | High risk — potentially harmful clause |

In [7]:
RISK_PROMPT = """You are a legal risk analyst. Score the following contract clause from 1 to 10.

CLAUSE TYPE: {clause_type}
EXTRACTED TEXT: {clause_text}

SIMILAR CLAUSES FROM OTHER CONTRACTS:
{rag_context}

Scoring guide:
1-3 = Low risk (standard clause, protects both parties equally)
4-6 = Medium risk (unusual terms, may need negotiation)
7-10 = High risk (one-sided, potentially harmful, missing key protections)

Respond in this exact format:
SCORE: <number 1-10>
REASONING: <one sentence explanation>
"""


def rag_search(query: str, clause_type: str = None, k: int = 3) -> str:
    """Search Qdrant for similar clauses, optionally filtered by type."""
    try:
        qvec = embedder.embed_query(query)
        filt = None
        if clause_type:
            filt = Filter(must=[FieldCondition(
                key='contract_type', match=MatchValue(value=clause_type)
            )])
        results = qdrant_client.query_points(
            collection_name=COLLECTION,
            query=qvec,
            query_filter=filt,
            limit=k,
            with_payload=True
        )
        if not results.points:
            return 'No similar clauses found in database.'
        return '\n---\n'.join(
            f"[{p.payload.get('source_file','?')[:40]}]\n{p.payload.get('text','')[:200]}"
            for p in results.points
        )
    except Exception as e:
        return f'RAG search error: {e}'


def parse_score_response(response: str) -> tuple:
    """Extract score and reasoning from LLM response."""
    score = 5  # default
    reasoning = response.strip()
    m = re.search(r'SCORE:\s*(\d+)', response, re.IGNORECASE)
    if m:
        score = min(10, max(1, int(m.group(1))))
    m2 = re.search(r'REASONING:\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    if m2:
        reasoning = m2.group(1).strip()[:300]
    return score, reasoning


class RiskScorerAgent:
    """Agent that scores legal risk for each extracted entity type."""

    CLAUSE_RISK_BASELINE = {
        'Termination':      6,
        'Indemnification':  7,
        'Non_Compete':      8,
        'IP_Ownership':     7,
        'Confidentiality':  5,
        'Governing_Law':    4,
        'Parties':          2,
        'Agreement_Date':   1,
    }

    def __init__(self, llm, embedder):
        self.llm = llm
        self.embedder = embedder

    def score_clause(self, clause_type: str, clause_texts: list) -> dict:
        clause_text = '; '.join(clause_texts[:3])  # take up to 3 spans
        rag_ctx = rag_search(clause_text, k=2)
        prompt = RISK_PROMPT.format(
            clause_type=clause_type,
            clause_text=clause_text[:400],
            rag_context=rag_ctx[:600],
        )
        try:
            resp = self.llm.invoke(prompt)
            score, reasoning = parse_score_response(resp)
        except Exception:
            score = self.CLAUSE_RISK_BASELINE.get(clause_type, 5)
            reasoning = 'LLM unavailable — using baseline heuristic score.'
        return {
            'score':     score,
            'reasoning': reasoning,
            'text':      clause_text[:200],
            'rag_used':  rag_ctx[:100] + '...'
        }

    def run(self, entities: dict) -> dict:
        print(f'[RiskScorerAgent] Scoring {len(entities)} clause types...')
        report = {}
        for clause_type, spans in entities.items():
            print(f'  Scoring: {clause_type}...')
            report[clause_type] = self.score_clause(clause_type, spans)
        return report


risk_scorer = RiskScorerAgent(llm=llm, embedder=embedder)
print('Both agents initialized!')
print(f'  ExtractorAgent  → tool: {extractor.tool_name}')
print(f'  RiskScorerAgent → LLM: flan-t5-base + Qdrant RAG')

Both agents initialized!
  ExtractorAgent  → tool: extract_legal_entities
  RiskScorerAgent → LLM: flan-t5-base + Qdrant RAG


## 7 — Full Agent Pipeline

The two agents work in sequence:
```
contract_text
      │
      ▼
ExtractorAgent.run()   ← calls BERT NER tool
      │
      ▼  entities dict
RiskScorerAgent.run()  ← calls RAG + LLM per clause
      │
      ▼
risk_report            ← {clause: {score, reasoning, text}}
```

In [8]:
def analyze_contract(contract_text: str, contract_name: str = 'Contract') -> dict:
    """Full pipeline: text -> entities -> risk scores."""
    print('=' * 65)
    print(f'ANALYZING: {contract_name}')
    print('=' * 65)

    # Step 1: Extractor Agent
    print('\nSTEP 1: Extractor Agent running BERT NER...')
    entities = extractor.run(contract_text)

    if not entities:
        print('No entities found — using sample entity for demo.')
        entities = {'Termination': ['termination on 30 days notice'],
                    'Governing_Law': ['State of Delaware']}

    print(f'\nExtracted entities:')
    for etype, spans in entities.items():
        print(f'  [{etype}]: {spans[:2]}')

    # Step 2: Risk Scorer Agent
    print('\nSTEP 2: Risk Scorer Agent running RAG + LLM scoring...')
    risk_report = risk_scorer.run(entities)

    return {
        'contract': contract_name,
        'entities': entities,
        'risk_report': risk_report,
    }

print('Pipeline ready!')

Pipeline ready!


## 8 — Display Risk Report

In [9]:
def print_risk_report(result: dict):
    entities    = result['entities']
    risk_report = result['risk_report']

    RISK_COLORS = {range(1,4): 'LOW', range(4,7): 'MEDIUM', range(7,11): 'HIGH'}

    def risk_label(score):
        for r, label in RISK_COLORS.items():
            if score in r:
                return label
        return 'MEDIUM'

    print('\n' + '=' * 65)
    print(f'  RISK REPORT: {result["contract"]}')
    print('=' * 65)

    sorted_items = sorted(risk_report.items(), key=lambda x: -x[1]['score'])
    for clause, data in sorted_items:
        label = risk_label(data['score'])
        bar   = '#' * data['score'] + '-' * (10 - data['score'])
        print(f'\n  {clause}')
        print(f'  Score : [{bar}] {data["score"]}/10  ({label})')
        print(f'  Text  : {data["text"][:100]}...')
        print(f'  Why   : {data["reasoning"][:150]}')

    all_scores = [d['score'] for d in risk_report.values()]
    avg = sum(all_scores) / len(all_scores) if all_scores else 0
    print('\n' + '=' * 65)
    print(f'  OVERALL RISK SCORE: {avg:.1f}/10  ({risk_label(int(avg))})')
    print('=' * 65)

## 9 — Test on Real CUAD Contract

In [10]:
from pathlib import Path

sample_path = sorted(Path('../data/sample_contracts').glob('*.txt'))[4]  # LIMEENERGYCO
sample_text = sample_path.read_text(encoding='utf-8')[:3000]  # first 3000 chars

print(f'Contract: {sample_path.name}')
print(f'Length: {len(sample_text)} chars')
print(f'Preview: {sample_text[:300]}...')

Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMENT.txt
Length: 3000 chars
Preview: EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

        ...


In [11]:
result = analyze_contract(sample_text, contract_name=sample_path.stem)
print_risk_report(result)

ANALYZING: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMENT

STEP 1: Extractor Agent running BERT NER...
[ExtractorAgent] Calling tool: extract_legal_entities
[ExtractorAgent] Extracted 0 entity types
No entities found — using sample entity for demo.

Extracted entities:
  [Termination]: ['termination on 30 days notice']
  [Governing_Law]: ['State of Delaware']

STEP 2: Risk Scorer Agent running RAG + LLM scoring...
[RiskScorerAgent] Scoring 2 clause types...
  Scoring: Termination...


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Scoring: Governing_Law...

  RISK REPORT: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMENT

  Termination
  Score : [#####-----] 5/10  (MEDIUM)
  Text  : termination on 30 days notice...
  Why   : <one sentence explanation>

  Governing_Law
  Score : [#####-----] 5/10  (MEDIUM)
  Text  : State of Delaware...
  Why   : <one sentence explanation>

  OVERALL RISK SCORE: 5.0/10  (MEDIUM)


## 10 — Test on Custom Contract Snippet

In [12]:
custom = """
CONSULTING AGREEMENT

This Agreement is made between TechCorp Inc. (Company) and John Doe (Consultant)
as of March 15, 2023.

TERM AND TERMINATION: Either party may terminate this Agreement immediately
without cause or written notice. Company reserves the right to withhold final
payment upon termination.

INDEMNIFICATION: Consultant shall indemnify and hold harmless the Company from
any and all claims, liabilities, and expenses including attorney fees.

INTELLECTUAL PROPERTY: All work product, inventions, and deliverables created
by Consultant shall be the exclusive property of Company.

NON-COMPETE: Consultant agrees not to work for any competitor for 5 years
anywhere in the world after termination.

GOVERNING LAW: This Agreement is governed by the laws of the Cayman Islands.
"""

result2 = analyze_contract(custom, contract_name='Custom_Consulting_Agreement')
print_risk_report(result2)

ANALYZING: Custom_Consulting_Agreement

STEP 1: Extractor Agent running BERT NER...
[ExtractorAgent] Calling tool: extract_legal_entities


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ExtractorAgent] Extracted 0 entity types
No entities found — using sample entity for demo.

Extracted entities:
  [Termination]: ['termination on 30 days notice']
  [Governing_Law]: ['State of Delaware']

STEP 2: Risk Scorer Agent running RAG + LLM scoring...
[RiskScorerAgent] Scoring 2 clause types...
  Scoring: Termination...
  Scoring: Governing_Law...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  RISK REPORT: Custom_Consulting_Agreement

  Termination
  Score : [#####-----] 5/10  (MEDIUM)
  Text  : termination on 30 days notice...
  Why   : <one sentence explanation>

  Governing_Law
  Score : [#####-----] 5/10  (MEDIUM)
  Text  : State of Delaware...
  Why   : <one sentence explanation>

  OVERALL RISK SCORE: 5.0/10  (MEDIUM)


## 11 — Save Risk Report to JSON

In [13]:
import json
from datetime import datetime

out_dir = Path('../data/risk_reports')
out_dir.mkdir(exist_ok=True)

for r in [result, result2]:
    fname = out_dir / f"{r['contract'][:40].replace(' ','_')}_{datetime.now():%Y%m%d_%H%M%S}.json"
    with open(fname, 'w') as f:
        json.dump(r, f, indent=2)
    print(f'Saved: {fname}')

Saved: ..\data\risk_reports\LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTO_20260607_123224.json
Saved: ..\data\risk_reports\Custom_Consulting_Agreement_20260607_123224.json


---
✅ Done! Two agents fully operational — Extractor + Risk Scorer.